In [16]:
import numpy as np
import pandas as pd
import mlflow
import dagshub
from scipy.stats import skew

dagshub.init(repo_owner='lbegi23', repo_name='House-Prices', mlflow=True)

Initialized MLflow to track repo "lbegi23/House-Prices"

Repository lbegi23/House-Prices initialized!

In [17]:
fill_with_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtExposure',
    'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1'
]
fill_with_0 = ['MasVnrArea']

In [18]:
def clean_and_engineer(df, skewed_cols=None, drop_outliers=False):
    df = df.copy()
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    df[fill_with_none] = df[fill_with_none].fillna('None')
    df[fill_with_0] = df[fill_with_0].fillna(0)
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])
    if drop_outliers and 'SalePrice' in df.columns:
        df = df.drop(df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)].index)
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['WasRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    df['TotalBath'] = (
        df['FullBath'] + df['HalfBath'] * 0.5 +
        df['BsmtFullBath'] + df['BsmtHalfBath'] * 0.5
    )
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)
    if skewed_cols is None:
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
        skewed = df[numeric_cols].apply(lambda x: skew(x.dropna()))
        skewed_cols = skewed[skewed > 0.75].index.tolist()
    cols_to_transform = [col for col in skewed_cols if col in df.columns and col != 'SalePrice']
    df[cols_to_transform] = np.log1p(df[cols_to_transform])
    return df, skewed_cols

In [19]:
train_df = pd.read_csv('./data/train.csv')
test_df = pd.read_csv('./data/test.csv')
test_ids = test_df['Id'].copy()

train_df, skewed_cols = clean_and_engineer(train_df, drop_outliers=True)
test_df, _ = clean_and_engineer(test_df, skewed_cols=skewed_cols)

train_df = pd.get_dummies(train_df, drop_first=True)
test_df = pd.get_dummies(test_df, drop_first=True)

X_train_full = train_df.drop(columns=['SalePrice', 'Id'], errors='ignore')

corr_matrix = X_train_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X_train_full = X_train_full.drop(columns=to_drop)

test_df = test_df.drop(columns=to_drop, errors='ignore')
test_df = test_df.reindex(columns=X_train_full.columns, fill_value=0)

model = mlflow.sklearn.load_model('models:/house-prices-xgboost-final/latest')

model_features = list(model.feature_names_in_)
X_test_kaggle = test_df.reindex(columns=model_features, fill_value=0)

preds = np.expm1(np.expm1(model.predict(X_test_kaggle)))

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds
})
submission.to_csv('./data/submission.csv', index=False)
print(submission.head())

     Id      SalePrice
0  1461  100999.671875
1  1462  131415.265625
2  1463  160768.687500
3  1464  157088.500000
4  1465  136502.718750


In [20]:
print(submission['SalePrice'].describe())

count      1459.000000
mean     139640.140625
std       52580.007812
min       42750.042969
25%      105041.421875
50%      127901.757812
75%      163817.687500
max      429940.750000
Name: SalePrice, dtype: float64
